# Agentes Baseados em Objetivos com Busca em Largura e Busca em Profundidade

Este notebook apresenta um tutorial prático sobre **agentes baseados em objetivos**.
A ideia central é simples: o agente observa o estado atual, define um objetivo e usa um algoritmo de busca para encontrar uma sequência de ações que leve até esse objetivo.

Aqui vamos:

1. revisar o papel de um agente baseado em objetivos;
2. implementar **busca em largura (BFS)** e **busca em profundidade (DFS)**;
3. comparar os dois algoritmos em um problema de rotas;
4. conectar a busca a um agente que planeja seus passos.

## Setup

Vamos reutilizar estruturas já existentes no projeto `aima-python`, em especial:

- `Problem` e `Node`, para representar problemas de busca;
- `GraphProblem`, para problemas em grafos;
- `romania_map`, um mapa clássico usado no livro;
- `SimpleProblemSolvingAgentProgram`, uma base para agentes que formulam objetivos e problemas.

In [1]:
from collections import deque

from search import (
    GraphProblem,
    Node,
    SimpleProblemSolvingAgentProgram,
    romania_map,
    breadth_first_graph_search,
    depth_first_graph_search,
)

## O que é um agente baseado em objetivos?

Um agente baseado em objetivos não reage apenas ao percepto imediato.
Ele também considera **onde quer chegar**.

O fluxo geral é:

1. atualizar sua representação interna do estado;
2. formular um objetivo;
3. formular um problema de busca;
4. executar uma busca;
5. seguir o plano gerado.

No contexto deste notebook, o estado será a **cidade atual** e o objetivo será **chegar a uma cidade destino**.

## Problema de exemplo: rotas no mapa da Romênia

Vamos usar o mapa da Romênia, em que:

- cada cidade é um estado;
- cada deslocamento possível é uma ação;
- o objetivo é sair de uma cidade inicial e chegar a uma cidade destino.

In [2]:
problema = GraphProblem('Arad', 'Bucharest', romania_map)

print('Estado inicial:', problema.initial)
print('Objetivo:', problema.goal)
print('Ações possíveis em Arad:', problema.actions('Arad'))

Estado inicial: Arad
Objetivo: Bucharest
Ações possíveis em Arad: ['Zerind', 'Sibiu', 'Timisoara']


## Funções auxiliares

As funções abaixo ajudam a reconstruir o caminho encontrado e a mostrar resultados de forma mais legível.

In [ ]:
def caminho_estados(node):
    if node is None:
        return None
    return [passo.state for passo in node.path()]


def resumo_busca(node, expansoes):
    if node is None:
        return {
            'encontrou': False,
            'caminho': None,
            'acoes': None,
            'custo': None,
            'expansoes': expansoes,
        }
    return {
        'encontrou': True,
        'caminho': caminho_estados(node),
        'acoes': node.solution(),
        'custo': node.path_cost,
        'expansoes': expansoes,
    }

## Implementação didática da busca em largura

A **busca em largura** expande primeiro os nós mais rasos da árvore de busca.
Em grafos não ponderados, ela encontra uma solução com o menor número de passos.

A implementação abaixo segue a ideia de `breadth_first_graph_search` do projeto, mas adiciona uma lista com a ordem de expansão para fins didáticos.

In [ ]:
def busca_em_largura_grafo(problem):
    node = Node(problem.initial)
    if problem.goal_test(node.state):
        return node, [node.state]

    fronteira = deque([node])
    explorados = set()
    expansoes = []

    while fronteira:
        node = fronteira.popleft()
        expansoes.append(node.state)
        explorados.add(node.state)

        for filho in node.expand(problem):
            if filho.state not in explorados and filho not in fronteira:
                if problem.goal_test(filho.state):
                    expansoes.append(filho.state)
                    return filho, expansoes
                fronteira.append(filho)

    return None, expansoes

## Implementação didática da busca em profundidade

A **busca em profundidade** expande primeiro os nós mais profundos.
Ela pode encontrar uma solução rapidamente em alguns casos, mas não garante o caminho com menos passos.

Aqui usamos a versão em grafo para evitar ciclos infinitos.

In [ ]:
def busca_em_profundidade_grafo(problem):
    fronteira = [Node(problem.initial)]
    explorados = set()
    expansoes = []

    while fronteira:
        node = fronteira.pop()
        expansoes.append(node.state)

        if problem.goal_test(node.state):
            return node, expansoes

        explorados.add(node.state)
        fronteira.extend(
            filho
            for filho in node.expand(problem)
            if filho.state not in explorados and filho not in fronteira
        )

    return None, expansoes

## Comparando BFS e DFS no mesmo problema

Vamos resolver o problema de ir de `Arad` até `Bucharest` com as duas estratégias.

In [ ]:
problema = GraphProblem('Arad', 'Bucharest', romania_map)

resultado_bfs, expansoes_bfs = busca_em_largura_grafo(problema)
resultado_dfs, expansoes_dfs = busca_em_profundidade_grafo(problema)

print('BFS')
print(resumo_busca(resultado_bfs, expansoes_bfs))
print()
print('DFS')
print(resumo_busca(resultado_dfs, expansoes_dfs))

### Leitura dos resultados

Observe principalmente:

- `caminho`: sequência de cidades visitadas até o objetivo;
- `acoes`: ações escolhidas pelo agente de busca;
- `custo`: soma das distâncias na rota encontrada;
- `expansoes`: ordem em que os nós foram retirados da fronteira para exploração.

Mesmo quando ambos encontram uma solução, o comportamento pode ser bem diferente.

In [ ]:
print('Caminho BFS:', caminho_estados(resultado_bfs))
print('Custo BFS:', resultado_bfs.path_cost)
print('Expansões BFS:', expansoes_bfs)
print()
print('Caminho DFS:', caminho_estados(resultado_dfs))
print('Custo DFS:', resultado_dfs.path_cost)
print('Expansões DFS:', expansoes_dfs)

## Usando as implementações prontas do projeto

O projeto já oferece versões prontas dessas buscas.
Podemos verificar que elas retornam soluções compatíveis com as versões didáticas implementadas acima.

In [ ]:
resultado_bfs_projeto = breadth_first_graph_search(problema)
resultado_dfs_projeto = depth_first_graph_search(problema)

print('BFS do projeto:', caminho_estados(resultado_bfs_projeto), '| custo =', resultado_bfs_projeto.path_cost)
print('DFS do projeto:', caminho_estados(resultado_dfs_projeto), '| custo =', resultado_dfs_projeto.path_cost)

## Transformando a busca em um agente

Agora vamos criar um agente concreto. Ele vai:

1. receber como percepto a cidade atual;
2. fixar um objetivo de destino;
3. formular um `GraphProblem`;
4. usar um algoritmo de busca para montar um plano.

Como em `GraphProblem` a ação é o nome da próxima cidade, o plano gerado já pode ser executado diretamente.

In [ ]:
class AgenteDeRotasBaseadoEmObjetivos(SimpleProblemSolvingAgentProgram):
    def __init__(self, objetivo, algoritmo_de_busca):
        super().__init__(initial_state=None)
        self.objetivo = objetivo
        self.algoritmo_de_busca = algoritmo_de_busca

    def update_state(self, state, percept):
        return percept

    def formulate_goal(self, state):
        return self.objetivo

    def formulate_problem(self, state, goal):
        return GraphProblem(state, goal, romania_map)

    def search(self, problem):
        resultado = self.algoritmo_de_busca(problem)
        if resultado is None:
            return []
        return resultado.solution()

## Simulação do agente

A função abaixo chama o agente repetidamente até ele chegar ao objetivo ou não ter mais ações.

In [ ]:
def simular_agente(agente, inicio, max_passos=20):
    estado = inicio
    trajetoria = [estado]
    acoes = []

    for _ in range(max_passos):
        acao = agente(estado)
        if acao is None:
            break
        acoes.append(acao)
        estado = acao
        trajetoria.append(estado)
        if estado == agente.objetivo:
            break

    return acoes, trajetoria

In [ ]:
agente_bfs = AgenteDeRotasBaseadoEmObjetivos('Bucharest', breadth_first_graph_search)
agente_dfs = AgenteDeRotasBaseadoEmObjetivos('Bucharest', depth_first_graph_search)

acoes_bfs, trajetoria_bfs = simular_agente(agente_bfs, 'Arad')
acoes_dfs, trajetoria_dfs = simular_agente(agente_dfs, 'Arad')

print('Agente com BFS')
print('Ações:', acoes_bfs)
print('Trajetória:', trajetoria_bfs)
print()
print('Agente com DFS')
print('Ações:', acoes_dfs)
print('Trajetória:', trajetoria_dfs)

## Discussão

Neste exemplo:

- o **objetivo** é chegar a `Bucharest`;
- o **problema** é navegar no grafo de cidades;
- a **busca em largura** tende a encontrar rotas com menos passos;
- a **busca em profundidade** pode seguir um caminho mais fundo antes de considerar alternativas.

Isso mostra a separação entre duas partes importantes:

- a definição do problema pelo agente;
- o algoritmo usado para procurar a solução.

## Exercícios sugeridos

1. Troque a cidade inicial para `Oradea` e compare BFS e DFS.
2. Mude o objetivo para `Neamt` e observe a ordem de expansão.
3. Teste `breadth_first_tree_search` e `depth_first_tree_search` e compare com as versões em grafo.
4. Adapte o agente para receber o objetivo como parâmetro em cada nova execução.